# LLM Missing Values and Outliers

## Import Libraries

In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
from utils.loader import load_tables
import json

BASE_LLM_MISSING_OUTPUT_PATH = MISSING_REPORT_DIR / "LLM"
BASE_LLM_OUTLIER_OUTPUT_PATH = OUTLIER_REPORT_DIR / "LLM"
os.makedirs(BASE_LLM_MISSING_OUTPUT_PATH, exist_ok=True)
os.makedirs(BASE_LLM_OUTLIER_OUTPUT_PATH, exist_ok=True)

dfs = load_tables("reconciled", normalize_cols="lower")

print("Libraries imported from utils. LLM client ready.")
print(f"LLM provider: {LLM_PROVIDER.upper()}")


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")

## LLM Missingness and Outliers

In [ ]:
for table_name, df in dfs.items():
    if df is None or len(df) == 0:
        print(f"[{table_name}] SKIPPED: empty dataframe.")
        continue

    print(f"\n{'='*60}\nLLM (MISSING/OUTLIERS) FOR: {table_name.upper()}\n{'='*60}")

    df = df.copy()
    df.columns = df.columns.str.lower()
    config = ANALYTICS_CONFIG.get(table_name, {})
    # 1. Summary extraction of actual missingness
    missing_counts = df.isnull().sum()
    cols_with_missing = missing_counts[missing_counts > 0].to_dict()

    miss_summary = {"table": table_name, "total_rows": len(df), "columns_with_missing": cols_with_missing}

    # ---------------------------------------------------------
    # PHASE A: Missingness Interpretation
    # ---------------------------------------------------------
    if cols_with_missing:
        print(f"[{table_name}] 1/3 Interpretation of Missing Values...")
        SYSTEM_MISSINGNESS = """You are a Data Scientist specializing in aviation databases. 
        Interpret statistical missingness in tables like Flights, Weather, Stations, or Aircraft.
        Explain if it's MCAR, MAR, or MNAR based on real-world aviation logic. Recommend a DW strategy (impute, drop, flag)."""

        PROMPT_MISSINGNESS = f"""Review the missingness statistics for our '{table_name}' table.
        MISSING SUMMARY: {json.dumps(miss_summary, indent=2)}
        
        Provide a structured interpretation:
        1. Classification: MCAR, MAR, or MNAR per column?
        2. Operational Context: Why are these values missing in real-world airline operations?
        3. Strategy: Recommend ETL handling."""

        missingness_report = call_llm(PROMPT_MISSINGNESS, system=SYSTEM_MISSINGNESS)
        if is_llm_error(missingness_report):
            print(f"[Missingness {table_name}] skipped -> {missingness_report}")
            missingness_report = ""
        else:
            with open(BASE_LLM_MISSING_OUTPUT_PATH / f"Missingness_{table_name}.md", "w", encoding="utf-8") as f:
                f.write("# System Prompt\n\n")
                f.write(SYSTEM_MISSINGNESS.strip())
                f.write("\n\n# User Prompt\n\n")
                f.write(PROMPT_MISSINGNESS.strip())
                f.write("\n\n# LLM Output\n\n")
                f.write(missingness_report)

        # ---------------------------------------------------------
        # PHASE B: Imputation Code Generation
        # ---------------------------------------------------------
        print(f"[{table_name}] 2/3 Generazione Codice di Imputazione Pandas...")
        SYSTEM_IMPUTATION = """You are a Data Engineering expert.
        Generate production-ready Python pandas code to handle missing values based on domain logic.
        Return ONLY valid Python code containing a function `impute_data(df)`. No prose outside comments."""

        PROMPT_IMPUTATION = f"""Write a Python function `impute_data_{table_name.lower()}(df)` that handles missing values for the {table_name} DataFrame.
        Columns requiring imputation: {list(cols_with_missing.keys())}.
        Use appropriate aviation/meteorological logic (e.g., median by category, forward-fill for weather, 0 for cancelled flights).
        Include before/after print statements.
        Return ONLY the python code block."""

        imputation_code = call_llm(PROMPT_IMPUTATION, system=SYSTEM_IMPUTATION)
        if is_llm_error(imputation_code):
            print(f"[Imputation {table_name}] skipped -> {imputation_code}")
        else:
            with open(BASE_LLM_MISSING_OUTPUT_PATH / f"Imputation_{table_name}.py", "w", encoding="utf-8") as f:
                f.write("# System Prompt:\n")
                f.write("# " + SYSTEM_IMPUTATION.strip().replace("\n", "\n# "))
                f.write("\n\n# User Prompt:\n")
                f.write("# " + PROMPT_IMPUTATION.strip().replace("\n", "\n# "))
                f.write("\n\n")
                f.write(imputation_code.replace("```python", "").replace("```", "").strip())
    else:
        print(f"[{table_name}] No missing values found. Skipping Phase A and B.")

    # ---------------------------------------------------------
    # PHASE C: Anomaly (Outlier) Report & ML Setup
    # ---------------------------------------------------------
    print(f"[{table_name}] 3/3 Outliers and ML Parameters Analysis...")
    configured_uni = [c[0] if isinstance(c, tuple) else c for c in config.get("outlier_uni", [])]
    configured_multi = config.get("outlier_multi", [])
    numeric_cols = [c for c in dict.fromkeys(configured_uni + configured_multi) if c in df.columns]

    if numeric_cols:
        SYSTEM_OUTLIER_EXPLAIN = """You are an Aviation Operations Analyst. 
        Translate quantitative outliers into operational realities (e.g., ATC delays, storms, sensor failures, typos)."""

        PROMPT_OUTLIER_EXPLAIN = f"""We are analyzing numeric extremes in the '{table_name}' table.
        Report-relevant columns selected for anomaly detection: {numeric_cols}.
        
        Provide:
        1. Likely Root Causes: What real-world events cause extreme high/low values in these specific columns?
        2. Data Validity: Are these typically valid extremes or data entry errors?
        3. Impact on Analytics: How do they skew our reporting if ignored?"""

        outlier_report = call_llm(PROMPT_OUTLIER_EXPLAIN, system=SYSTEM_OUTLIER_EXPLAIN)
        if is_llm_error(outlier_report):
            print(f"[Outliers {table_name}] skipped -> {outlier_report}")
            outlier_report = ""
        else:
            with open(BASE_LLM_OUTLIER_OUTPUT_PATH / f"Outliers_{table_name}.md", "w", encoding="utf-8") as f:
                f.write("# System Prompt\n\n")
                f.write(SYSTEM_OUTLIER_EXPLAIN.strip())
                f.write("\n\n# User Prompt\n\n")
                f.write(PROMPT_OUTLIER_EXPLAIN.strip())
                f.write("\n\n# LLM Output\n\n")
                f.write(outlier_report)

        # ML Advisor (Isolation Forest)
        SYSTEM_ML_ADVISOR = """You are an ML Engineer. Recommend parameters for Isolation Forest. Focus on 'contamination'."""
        PROMPT_ML_ADVISOR = f"""We want to apply Isolation Forest to the '{table_name}' table on these report-relevant columns: {numeric_cols}.
        Recommend the 'contamination' parameter. Justify your choice based on typical anomaly frequencies in this domain."""

        ml_recommendations = call_llm(PROMPT_ML_ADVISOR, system=SYSTEM_ML_ADVISOR)
        if is_llm_error(ml_recommendations):
            print(f"[ML Config {table_name}] skipped -> {ml_recommendations}")
        else:
            with open(BASE_LLM_OUTLIER_OUTPUT_PATH / f"ML_Config_{table_name}.md", "w", encoding="utf-8") as f:
                f.write("# System Prompt\n\n")
                f.write(SYSTEM_ML_ADVISOR.strip())
                f.write("\n\n# User Prompt\n\n")
                f.write(PROMPT_ML_ADVISOR.strip())
                f.write("\n\n# LLM Output\n\n")
                f.write(ml_recommendations)
    else:
        print(f"[{table_name}] No report-relevant numeric columns configured for outlier analysis.")

print("\nLLM-Supported Data Quality Analysis Completed.")